In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
os.makedirs("../images", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# Load raw dataset from data/raw/
raw_path = "../data/raw/ott_movie_data.csv"
df = pd.read_csv(raw_path)
print("Loaded raw shape:", df.shape)

In [ ]:
display(df.head())
print("\n--- Dataset Info ---")
df.info()

print("\n--- Numeric Summary ---")
display(df.describe())

print("\n--- Missing Values per Column ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows Count ---")
print("Exact duplicates:", df.duplicated().sum())

print("\n--- Unique Values in Category Columns ---")
for col in ["City", "Subscription_Tier", "Content_Type", "Device_Type"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

In [ ]:
df_clean = df.drop_duplicates().copy()
print("Rows after dropping duplicates:", len(df_clean))

# Clean inconsistent text in Content_Type
df_clean["Content_Type"] = (
    df_clean["Content_Type"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Convert User_Age to numeric and replace impossible ages (<18 or >60) with NaN
df_clean["User_Age"] = pd.to_numeric(df_clean["User_Age"], errors="coerce")
df_clean.loc[(df_clean["User_Age"] < 18) | (df_clean["User_Age"] > 60), "User_Age"] = np.nan

df_clean["Buffering_Events"] = pd.to_numeric(df_clean["Buffering_Events"], errors="coerce")

# Impute missing numeric values with median
for col in ["User_Age", "Buffering_Events"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Impute missing categorical values with mode
for col in ["City", "Device_Type"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Ensure watch duration never exceeds runtime
df_clean["Watch_Duration_Mins"] = np.minimum(
    df_clean["Watch_Duration_Mins"], df_clean["Content_Runtime_Mins"]
)

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())
print("Cleaned shape:", df_clean.shape)

In [ ]:
df_clean["Completion_Rate_Pct"] = (
    (df_clean["Watch_Duration_Mins"] / df_clean["Content_Runtime_Mins"]) * 100
).round(1)

df_clean["Engagement_Status"] = np.where(
    df_clean["Watch_Duration_Mins"] < 30,
    "Drop-Off Risk (<30m)",
    np.where(
        df_clean["Watch_Duration_Mins"] <= 75,
        "Moderate Watch (30-75m)",
        "High Engagement (>75m)",
    ),
)

df_clean["Age_Group"] = pd.cut(
    df_clean["User_Age"],
    bins=[17, 25, 40, 60],
    labels=["Young Adult (18-25)", "Working Adult (26-40)", "Mature Viewer (41-60)"],
)

display(
    df_clean[
        [
            "Session_ID",
            "Watch_Duration_Mins",
            "Content_Runtime_Mins",
            "Completion_Rate_Pct",
            "Engagement_Status",
            "User_Age",
            "Age_Group",
        ]
    ].head()
)

In [ ]:
q1_stats = df_clean["Watch_Duration_Mins"].agg(["count", "mean", "median", "std", "min", "max"])
print("Q1 - Overall Watch Duration Stats:")
print(q1_stats.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df_clean["Watch_Duration_Mins"], bins=20, kde=True, color="#2563EB", ax=ax)
ax.axvline(q1_stats["mean"], color="#DC2626", linestyle="--", linewidth=1.8, label=f"Mean: {q1_stats['mean']:.1f} min")
ax.axvline(q1_stats["median"], color="#16A34A", linestyle="-.", linewidth=1.8, label=f"Median: {q1_stats['median']:.1f} min")
ax.axvline(30, color="#F59E0B", linestyle=":", linewidth=2, label="Drop-Off Threshold (30 min)")
ax.set_title("Distribution of Session Watch Duration (Q1)", fontsize=13, fontweight="bold")
ax.set_xlabel("Watch Duration (Minutes)")
ax.set_ylabel("Number of Sessions")
ax.legend()
plt.tight_layout()
plt.savefig("../images/q1_watch_duration_distribution.png", dpi=200)
plt.show()

In [ ]:
corr_matrix = df_clean[
    ["Watch_Duration_Mins", "Content_Runtime_Mins", "Buffering_Events", "User_Age", "Trailer_Clicks"]
].corr()["Watch_Duration_Mins"]
print("Q2 - Correlation with Watch_Duration_Mins:")
print(corr_matrix.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.regplot(
    data=df_clean,
    x="Content_Runtime_Mins",
    y="Watch_Duration_Mins",
    scatter_kws={"alpha": 0.6, "color": "#2563EB"},
    line_kws={"color": "#DC2626"},
    ax=axes[0],
)
axes[0].set_title(f"Runtime vs. Watch Duration (r = {corr_matrix['Content_Runtime_Mins']:.2f})", fontweight="bold")
axes[0].set_xlabel("Content Runtime (Minutes)")
axes[0].set_ylabel("Watch Duration (Minutes)")

sns.boxplot(data=df_clean, x="Buffering_Events", y="Watch_Duration_Mins", palette="Blues", ax=axes[1])
axes[1].set_title(f"Buffering Events vs. Watch Duration (r = {corr_matrix['Buffering_Events']:.2f})", fontweight="bold")
axes[1].set_xlabel("Buffering Events Count")
axes[1].set_ylabel("Watch Duration (Minutes)")

plt.suptitle("Impact of Runtime and Buffering on Watch Duration (Q2)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../images/q2_runtime_and_buffering_impact.png", dpi=200)
plt.show()

In [ ]:
for col in ["Subscription_Tier", "Content_Type", "Device_Type", "City"]:
    print(f"\nQ3 - Mean Watch Duration by {col}:")
    display(
        df_clean.groupby(col, observed=False)["Watch_Duration_Mins"]
        .agg(["count", "mean", "median"])
        .round(2)
        .sort_values("mean", ascending=False)
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

tier_order = ["Mobile", "Basic", "Standard", "Premium"]
tier_means = df_clean.groupby("Subscription_Tier")["Watch_Duration_Mins"].mean().reindex(tier_order)
sns.barplot(x=tier_means.index, y=tier_means.values, palette="viridis", ax=axes[0])
axes[0].set_title("Mean Watch Duration by Subscription Tier", fontweight="bold")
axes[0].set_xlabel("Subscription Tier")
axes[0].set_ylabel("Mean Watch Duration (Minutes)")
for i, v in enumerate(tier_means.values):
    axes[0].text(i, v + 1.2, f"{v:.1f}m", ha="center", fontweight="bold")

type_means = df_clean.groupby("Content_Type")["Watch_Duration_Mins"].mean().sort_values(ascending=False)
sns.barplot(x=type_means.index, y=type_means.values, palette="magma", ax=axes[1])
axes[1].set_title("Mean Watch Duration by Content Type", fontweight="bold")
axes[1].set_xlabel("Content Type")
axes[1].set_ylabel("Mean Watch Duration (Minutes)")
for i, v in enumerate(type_means.values):
    axes[1].text(i, v + 1.2, f"{v:.1f}m", ha="center", fontweight="bold")

plt.suptitle("Watch Duration Across Subscription Tiers & Content Formats (Q3)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../images/q3_category_watch_duration.png", dpi=200)
plt.show()

In [ ]:
at_risk = df_clean[df_clean["Watch_Duration_Mins"] < 30]
print(f"Q4 - Drop-Off Risk Sessions (<30m): {len(at_risk)} out of {len(df_clean)} ({len(at_risk)/len(df_clean)*100:.1f}%)")
display(at_risk[["Session_ID", "City", "Subscription_Tier", "Content_Type", "Buffering_Events", "Watch_Duration_Mins"]].head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

status_order = ["Drop-Off Risk (<30m)", "Moderate Watch (30-75m)", "High Engagement (>75m)"]
status_counts = df_clean["Engagement_Status"].value_counts().reindex(status_order)
sns.barplot(x=status_counts.index, y=status_counts.values, palette=["#EF4444", "#F59E0B", "#10B981"], ax=axes[0])
axes[0].set_title("Sessions by Engagement Status", fontweight="bold")
axes[0].set_xlabel("Engagement Status")
axes[0].set_ylabel("Number of Sessions")
for i, v in enumerate(status_counts.values):
    pct = v / len(df_clean) * 100
    axes[0].text(i, v + 2, f"{v} ({pct:.1f}%)", ha="center", fontweight="bold")

risk_by_tier = (
    df_clean.assign(Is_Risk=df_clean["Watch_Duration_Mins"] < 30)
    .groupby("Subscription_Tier")["Is_Risk"]
    .mean()
    .reindex(tier_order)
    * 100
)
sns.barplot(x=risk_by_tier.index, y=risk_by_tier.values, palette="Reds", ax=axes[1])
axes[1].set_title("Drop-Off Risk Rate (<30m) by Subscription Tier", fontweight="bold")
axes[1].set_xlabel("Subscription Tier")
axes[1].set_ylabel("Share of Sessions Under 30 Mins (%)")
for i, v in enumerate(risk_by_tier.values):
    axes[1].text(i, v + 0.6, f"{v:.1f}%", ha="center", fontweight="bold")

plt.suptitle("Drop-Off Risk Identification (<30 Mins) (Q4)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../images/q4_dropoff_risk_analysis.png", dpi=200)
plt.show()

In [ ]:
clean_path = "../data/processed/ott_movie_data_clean.csv"
df_clean.to_csv(clean_path, index=False)
print("Saved cleaned dataset to:", clean_path)